In [17]:
# imports and env
import base64
import mimetypes
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_groq import ChatGroq


load_dotenv()

True

In [2]:
import sys 
print(sys.executable)

c:\Users\Vanilla\anaconda3\envs\myenv\python.exe


In [3]:
from unstructured.partition.pdf import partition_pdf
print("PDF dependencies loaded")

PDF dependencies loaded


In [4]:
# load PDF
PDF_PATH = "crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized


INFO: Reading PDF for file: crag_paper.pdf ...


Loaded 247 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 8
  ListItem: 41
  NarrativeText: 106
  Table: 7
  Title: 32
  UncategorizedText: 44


In [5]:
elements[0].metadata.keys()

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'filename', 'category', 'element_id'])

In [20]:
for element in elements:
    if element.metadata.get("category") == "Image":
        print(element.metadata.keys())
        print(element.metadata)
        break

dict_keys(['source', 'coordinates', 'last_modified', 'filetype', 'languages', 'page_number', 'image_path', 'filename', 'category', 'element_id'])
{'source': 'crag_paper.pdf', 'coordinates': {'points': ((np.float64(213.1065000888888), np.float64(1685.6428340266664)), (np.float64(213.1065000888888), np.float64(1761.8372080844445)), (np.float64(268.9823708577777), np.float64(1761.8372080844445)), (np.float64(268.9823708577777), np.float64(1685.6428340266664))), 'system': 'PixelSpace', 'layout_width': 1654, 'layout_height': 2339}, 'last_modified': '2026-08-24T09:09:50', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'image_path': 'g:\\sushant\\Advanced_RAG\\14.MultiModal_RAG\\figures\\figure-1-1.jpg', 'filename': 'crag_paper.pdf', 'category': 'Image', 'element_id': '9c5fdac1264f8a2f2f4ea2b1a0c7cda6'}


In [ ]:
# caption images with VLM
import os
vlm = ChatOpenAI(
    model="qwen/qwen3-vl-32b-instruct",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=1200,
)

IMAGE_CAPTION_SYSTEM_PROMPT = """You are a document analysis assistant. Your task is to generate \
detailed, accurate descriptions of images extracted from a document. These descriptions will be \
embedded into a vector store and used for semantic retrieval, so they must capture all information \
a user might search for.

For each image, describe:
- The image type (chart, diagram, photograph, table, illustration, screenshot, etc.)
- All visible text, labels, titles, captions, and annotations
- Key data, values, trends, or patterns (especially for charts and graphs)
- The main subject and all important visual elements
- Spatial relationships and structure where relevant

Be specific and thorough. Avoid vague language."""

def encode_image(image_path: str) -> str:
    """Convert the image from pixels to a base64 string for embedding in a prompt. As model only understands text and not binary data."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def caption_image(image_path: str) -> str:
    """Generate a detailed description of an image using a vision-language model (VLM). The description will be used for semantic retrieval in a vector store."""
    b64 = encode_image(image_path)
    messages = [
        SystemMessage(content=IMAGE_CAPTION_SYSTEM_PROMPT),
        HumanMessage(content=[
            {"type": "text", "text": "Describe this image extracted from a document."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}, ### for openai models
        ]),
    ]
    return vlm.invoke(messages).content

In [7]:
image_docs = []

for el in elements:
    if el.metadata.get("category") == "Image":
        image_path = el.metadata.get("image_path", "")
        if image_path:
            caption = caption_image(image_path)
            image_docs.append(Document(page_content=caption, metadata=el.metadata))

print(f"Captioned {len(image_docs)} images")

INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Captioned 8 images


In [8]:
print(image_docs[-1].page_content)

This image is a line chart comparing the **Accuracy of generation** (y-axis) against the **Accuracy of retrieval** (x-axis) for two different models: **Self-RAG** and **Self-CRAG**.

---

### **Chart Details**

- **Title/Context**: Not explicitly stated, but implied to be a performance comparison between Self-RAG and Self-CRAG under varying retrieval accuracy conditions.
  
- **X-axis (Horizontal)**:
  - Label: **Accuracy of retrieval**
  - Values range from 10 to 60 (in descending order from right to left).
  - Tick marks at intervals of 10: 10, 20, 30, 40, 50, 60.
  - Note: The x-axis is reversed — higher retrieval accuracy is on the left.

- **Y-axis (Vertical)**:
  - Label: **Accuracy of generation**
  - Values range from 20 to 70.
  - Tick marks at intervals of 10: 20, 30, 40, 50, 60, 70.

- **Legend**:
  - **Self-RAG**: Represented by green stars (`★`).
  - **Self-CRAG**: Represented by gray diamonds (`◆`).

- **Data Lines**:
  - **Self-RAG (green stars)**:
    - Starts at approx

In [9]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]

text_docs = splitter.split_documents(text_elements)
caption_docs = splitter.split_documents(image_docs)

all_docs = text_docs + caption_docs
print(f"Total chunks: {len(all_docs)} ({len(text_docs)} text + {len(caption_docs)} captions)")

Total chunks: 267 (249 text + 18 captions)


In [12]:
# embeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_ollama import ChatOllama, OllamaEmbeddings

embeddings = OllamaEmbeddings(model="qwen3-embedding:4b")

In [13]:
# vector store
vector_store = Chroma.from_documents(filter_complex_metadata(all_docs), embeddings)

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [14]:
# retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [20]:
# RAG chain
prompt = ChatPromptTemplate.from_messages([
    ("human", "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"),
])

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    messages = prompt.format_messages(context=text_context, question=question)

    ### Deduplication of image paths to avoid sending the same image multiple times in the prompt
    if image_chunks:
        seen_paths = set()
        image_content = []
        for doc in image_chunks:
            image_path = doc.metadata.get("image_path")
            if image_path and image_path not in seen_paths:
                seen_paths.add(image_path)
                image_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"},
                })

        if image_content:
            text_content = messages[-1].content
            messages[-1] = HumanMessage(content=[
                {"type": "text", "text": text_content},
                *image_content,
            ])

    return messages

# llm = ChatOpenAI(model="gpt-5-mini")

llm = ChatGroq(model="openai/gpt-oss-120b")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | {"response": vlm | StrOutputParser(), "context": RunnablePassthrough()}
)

In [21]:
# test
question = "How does Self-CRAG compares with Self-RAG as shown in the line chart. Can you explain this in a little bit more detail?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Based on the line chart, **Self-CRAG consistently outperforms Self-RAG** across all levels of retrieval accuracy, and this performance gap widens as retrieval accuracy decreases.

Here’s a more detailed explanation:

- The x-axis represents the **accuracy of retrieval**, ranging from 60 down to 10.
- The y-axis shows the **accuracy of generation** (i.e., how well the model generates correct responses).
- The dashed blue line at ~29.8 indicates the baseline performance with **no retrieval** — meaning if no relevant information is retrieved, the model relies solely on its internal knowledge.

### Key Observations:

1. **At High Retrieval Accuracy (e.g., 60)**:
   - Self-RAG starts at around 55% generation accuracy.
   - Self-CRAG starts at approximately 62%, which is significantly higher.
   - This suggests that even when retrieval is good, CRAG’s plug-and-play enhancement improves generation quality.

2. **As Retrieval Accuracy Decreases**:
   - Self-RAG’s generation accuracy drops stea

In [22]:
len(answer["context"][0].content)

2

In [23]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer["response"])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Based on the provided context:

- **CRAG** has a computational requirement of **27.2 TFLOPs per token** and an execution time of **0.512 seconds per instance**.
- **Self-RAG** has a computational requirement ranging from **26.5 to 132.4 TFLOPs per token** (a variable range) and an execution time of **0.741 seconds per instance**.

### Comparison:
- **Faster Execution Time**: **CRAG** is faster than Self-RAG (0.512s vs. 0.741s).
- **TFLOPs**: CRAG has a fixed, lower value (27.2 TFLOPs/token) compared to Self-RAG’s wider range (26.5–132.4 TFLOPs/token). The minimum for Self-RAG (26.5) is slightly lower than CRAG’s 27.2, but the maximum is significantly higher, indicating greater variability and potential for much higher computational cost.

Thus, **CRAG has faster execution time and more consistent, lower computational requirements** compared to Self-RAG.

> Note: The values are estimates for the generation phase only; retrieval and data-processing stages are excluded.
